# 10 임베딩 기반 시계열 분석 (Phase 2)

## 왜 XGBoost를 대표 base로 고정했는가?

Phase1(07–09) **10모델 통합 Best** 결과(35개 유효 조건, MAPE 기준)에서:
- **XGBoost 13회**로 Best 빈도 **1위** (2위 ARIMA 5회)
- SBC·ML **양쪽 scheme**에서 모두 자주 선택
- 임베딩 벡터를 **추가 피처**로 붙이기에 패널 ML 구조가 적합

조건마다 Best base가 다르면 `조건×서로 다른 base×6임베딩`으로 실험 수가 과도해지므로, **임베딩 방법(6종) 비교**를 목적으로 전체 최다 winner인 **XGBoost**를 대표 base로 고정합니다.

| 축 | 내용 |
|----|------|
| type | A, B, C, D, E (5) |
| cluster | SBC(rule-base) + ML(PatchTST+HAC), 각 4클러스터 |
| **조건** | **40** (5×4×2) |
| base (고정) | **XGBoost** |
| 임베딩 | PCA, FastDTW, AE, GAF-CNN, TS2Vec, PatchTST |
| 조합 | **40 × 6 = 240** |

**오차지표:** MAE, RMSE, MAPE, MASE — 조건별 Best 임베딩은 **MAPE** 최소 기준

## 11장과의 연결

본 장은 **동일 예측 파이프라인(XGBoost+임베딩)** 위에서 SBC·ML **클러스터링 scheme만** 바꿔 조건별 Best 임베딩을 산출합니다.  
11장에서는 이 예측을 type별 **가중 WMAPE**로 합산해 **어느 scheme이 유리한지** 판단하고, **변동계수(CV)** 와 대조해 논문 가설(고변동→SBC, 저변동→ML)과의 정합성을 검토합니다.

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())


Torch device: cuda (NVIDIA GeForce RTX 5090, 32GB)
시계열: 165
학습 <= 201730 | 검증: [201731, 201732, 201733]
Best 선정 기준: MAPE


### ① XGBoost × 6임베딩 — 40조건 실험

In [2]:
from utils.phase_experiments import REPRESENTATIVE_BASE_MODEL

BASE = REPRESENTATIVE_BASE_MODEL
print('대표 base:', BASE)

p2_cache = DATA_PROCESSED / 'phase2_results.parquet'
if p2_cache.exists():
    phase2 = pd.read_parquet(p2_cache)
    phase2_summary, phase2_best = summarize_phase2(phase2)
    print('Phase2 캐시 로드 |', len(phase2), 'rows')
else:
    emb_cache = build_global_embedding_cache(df)
    p2_sbc = run_phase2_all(df, feat_df, 'SBC_CLUSTER', 'SBC', fixed_model=BASE, emb_cache=emb_cache)
    p2_ml = run_phase2_all(df, feat_df, 'ML_CLUSTER', 'ML', fixed_model=BASE, emb_cache=emb_cache)
    phase2 = pd.concat([p2_sbc, p2_ml], ignore_index=True)
    phase2_summary, phase2_best = summarize_phase2(phase2)
    phase2.to_parquet(p2_cache, index=False)
    phase2_summary.to_csv(DATA_PROCESSED / 'phase2_summary.csv', index=False)
    phase2_best.to_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv', index=False)
    print('Phase2 완료 |', len(phase2), 'rows')

print('유효 조건(제품 있음):', phase2.groupby(['cluster_scheme','type','cluster']).ngroups)
display(phase2_best.sort_values(['cluster_scheme', 'type', 'cluster']))


대표 base: XGBoost


Global embeddings:   0%|          | 0/6 [00:00<?, ?it/s]

Phase2 SBC:   0%|          | 0/120 [00:00<?, ?it/s]

C:\Users\kjh\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py:751: UserWarning: [13:26:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Phase2 ML:   0%|          | 0/120 [00:00<?, ?it/s]

Phase2 완료 | 1980 rows
유효 조건(제품 있음): 34


,cluster_scheme,type,cluster,best_hybrid,base_model,embedding,mae_mean,rmse_mean,best_mape,mase_mean
2,ML,A,1,XGBoost+GAF-CNN,XGBoost,GAF-CNN,4910.632487,6749.678281,93.967943,4.094132
6,ML,A,2,XGBoost+AE,XGBoost,AE,140522.250000,177293.581182,112.903346,4.518172
12,ML,A,3,XGBoost+AE,XGBoost,AE,49848.974304,75119.140268,85.774429,2.744677
18,ML,A,4,XGBoost+AE,XGBoost,AE,150838.447917,202877.274264,96.230749,3.204246
28,ML,B,1,XGBoost+PatchTST,XGBoost,PatchTST,2764.447040,3785.017502,94.980155,2.893273
30,ML,B,2,XGBoost+AE,XGBoost,AE,56460.509458,87657.021497,64.616521,1.885630
36,ML,B,3,XGBoost+AE,XGBoost,AE,36043.276042,55611.716663,59.207989,2.606348
42,ML,C,1,XGBoost+AE,XGBoost,AE,3452.555374,4634.614353,144.160974,2.991508
48,ML,C,2,XGBoost+AE,XGBoost,AE,73467.885417,116602.826483,75.216892,3.166962
54,ML,C,3,XGBoost+AE,XGBoost,AE,60325.468750,73562.356450,62.630551,5.098773


### ② 결과 요약

### ③ 11장 하이브리드 비교로의 연결

- 본 장 산출물 `phase2_best_per_condition.csv` = 조건별 **XGBoost + Best 임베딩** (MAPE 기준)
- 11장에서 SBC·ML scheme 각각의 Best 임베딩 예측을 type별 **가중 WMAPE**로 합산
- **동일 base·동일 임베딩 파이프라인**에서 clustering scheme만 다르므로, SBC vs ML 비교가 공정함
- 11장에서 **CV(변동계수)** 와 대조 → 고변동 type(B,E)에서 SBC 우세 등, 논문 방법론의 **부분적 타당성** 검토 (단, 모든 데이터에 항상 성립하지는 않음)

In [3]:
print('=== 조건별 Best 임베딩 (MAPE) ===')
print(phase2_best['embedding'].value_counts())

print('\n=== 임베딩별 평균 4지표 (전 조건) ===')
emb_avg = phase2.groupby('embedding')[['mae','rmse','mape','mase']].mean().round(2)
display(emb_avg.sort_values('mape'))

print('\n=== scheme별 XGBoost+임베딩 평균 MAPE ===')
print(phase2.groupby(['cluster_scheme','embedding'])['mape'].mean().unstack('cluster_scheme').round(2))


=== 조건별 Best 임베딩 (MAPE) ===
embedding
AE          23
FastDTW      5
GAF-CNN      3
PatchTST     3
Name: count, dtype: int64

=== 임베딩별 평균 4지표 (전 조건) ===


,mae,rmse,mape,mase
embedding,,,,
AE,9245.79,12883.60,110.52,3.78
GAF-CNN,9232.62,12895.74,111.13,3.84
PCA,9232.62,12895.74,111.13,3.84
TS2Vec,9232.62,12895.74,111.13,3.84
FastDTW,9213.89,12866.15,120.95,3.73
PatchTST,9198.02,12888.85,123.23,3.93



=== scheme별 XGBoost+임베딩 평균 MAPE ===
cluster_scheme      ML     SBC
embedding                     
AE              109.72  111.31
FastDTW         112.31  129.59
GAF-CNN         113.04  109.23
PCA             113.04  109.23
PatchTST        126.77  119.70
TS2Vec          113.04  109.23
